In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import copy

## CellOracle
import celloracle as co

## Load data

We use CellOracle data: only CPM normalization without log transformation. All genes included.

=> we modify .raw information to have the CPM normalization instead of log transformed.

In [2]:
# ==================================================================
# LOAD DATA
# adata: contains UMAP, Leiden, adata.X = z-scored HVGs
# adata_CPM: log-normalized full genome (used as raw for CellOracle)
# base_GRN: base GRN with lncRNA and regulated genes included 
# ==================================================================

adata = sc.read_h5ad("../data/data_pseudotime.h5ad")

## Only CPM normalization in raw of adata. Also subset adata_CPM to the cells that don't
## belong to the 'noisy' clusters (filtered in DE expression analysis).
adata_CPM = sc.read_h5ad("../data/data_CPM_norm.h5ad")
adata_CPM = adata_CPM[adata.obs_names].copy()

## Base GRN
base_GRN = pd.read_parquet('../data/celloracle_data/GRN_v1_Both_Dense.parquet')

## Modify data for CellOracle use:

First we just consider the genes that exist both in the base GRN and the adata.raw object (scVASAseq data).

Secondly we look to perform the select a certain ammount of the data to be used, since CellOracle works well between 2000-3000 genes.

=> We keep TFs, lncRNA and DE genes in the network.

=> The rest of selected genes will be the HVG, priorizing the more variable one's.

In [3]:
## Total input genes for CellOracle
total_genes_to_use = 2100

# =============================================================
# STEP 1: FILTER GRN TO GENES PRESENT IN CPM DATA
# =============================================================

data_genes = set(adata_CPM.var_names)

base_GRN_filtered = base_GRN[
    base_GRN['source'].isin(data_genes) &
    base_GRN['target'].isin(data_genes)
].copy()

print(f"Base GRN BEFORE filtering to CPM data genes: {base_GRN.nunique()}")
print("    ")
print(f"Base GRN AFTER filtering to CPM data genes: {base_GRN_filtered.nunique()}")
print('=============================================================')

# =============================================================
# STEP 2: DEFINE MANDATORY GENES
# =============================================================

TFs_set          = set(base_GRN_filtered['source'].unique())
rmst1_targets    = set(base_GRN_filtered[base_GRN_filtered['source'] == 'Rmst']['target'])
c13_targets      = set(base_GRN_filtered[base_GRN_filtered['source'] == 'C130026L21Rik']['target'])
lncrna_set       = set(['Rmst', 'C130026L21Rik'] )

mandatory_genes  = TFs_set | rmst1_targets | c13_targets | lncrna_set

n_mandatory      = len(mandatory_genes)
n_extra_slots    = total_genes_to_use - n_mandatory

print(f"\nMandatory genes: {n_mandatory}")
print(f"  TFs: {len(TFs_set & data_genes)}")
print(f"  Rmst targets: {len(rmst1_targets)}")
print(f"  C13 targets: {len(c13_targets)}")
print(f"  lncRNAs: {len(lncrna_set)}")
print(f"Extra slots for HVGs: {n_extra_slots}")

if n_extra_slots < 0:
    print(f"WARNING: mandatory genes ({n_mandatory}) already exceed total genes ({total_genes_to_use}).")

print('=============================================================')

# =============================================================
# STEP 3: SELECT HVGs FROM LOG-TRANSFORMED CPM TO FILL EXTRA SLOTS
# We log-transform a copy of CPM only for HVG identification.
# The actual expression matrix will remain CPM (no log).
# =============================================================

# Protein-coding only, excluding ribosomal/mitochondrial noise
noise_prefixes = ('Rps', 'Rpl', 'mt-')
pc_mask = (
    adata_CPM.var.get('is_protein_coding') &
    ~pd.Index(adata_CPM.var_names).str.startswith(noise_prefixes)
)
adata_for_hvg = adata_CPM[:, pc_mask].copy()

# Log-transform CPM temporarily for HVG computation only
# We take more then filter in case there is overlapping with
# mandatory genes. 
sc.pp.log1p(adata_for_hvg)
sc.pp.highly_variable_genes(
    adata_for_hvg,
    n_top_genes=n_extra_slots + n_mandatory,
)

# HVGs ranked by dispersion, excluding mandatory genes already included
hvg_candidates = (
    adata_for_hvg.var[adata_for_hvg.var['highly_variable']]
    .sort_values('dispersions_norm', ascending=False)
    .index.tolist()
)
hvg_extra = [g for g in hvg_candidates if g not in mandatory_genes][:n_extra_slots]

print(f"HVG extra genes selected: {len(hvg_extra)}")
print('=============================================================')

# =============================================================
# STEP 4: BUILD FINAL GENE SET AND SUBSET CPM DATA
# =============================================================

final_genes = list(mandatory_genes | set(hvg_extra))
print(f"Final gene set size: {len(final_genes)}")

# Verify lncRNAs are in the final set
for name in lncrna_set:
    print(f"  {name} in final set: {name in final_genes}")
print('=============================================================')

# =========================================================================
# STEP 5: BUILD adata_celloracle
# X = CPM counts for final genes (Physics requirement: absolute magnitude)
# obsm/obs/uns = Topology from the fully processed adata
# =========================================================================

adata_celloracle = adata_CPM[:, final_genes].copy()

# Assign metadata to new object
adata_celloracle.obs = adata.obs.copy()
for key in adata.obsm.keys():
    adata_celloracle.obsm[key] = adata.obsm[key].copy()
# deepcopy prevents memory entanglement between the two objects
adata_celloracle.uns = copy.deepcopy(adata.uns)

# Sanity check
print(f"adata_celloracle shape: {adata_celloracle.shape}")
print(f"UMAP present: {'X_umap' in adata_celloracle.obsm}")
print(f"Leiden present: {'leiden_annotated' in adata_celloracle.obs}")
print('=============================================================')


# =============================================================
# STEP 6: BUILD TFdict FROM FILTERED GRN
# =============================================================

base_GRN_TFdict = (
    base_GRN_filtered[
        base_GRN_filtered['target'].isin(final_genes) &
        base_GRN_filtered['source'].isin(final_genes)
    ]
    .groupby('target')['source']
    .apply(list)
    .to_dict()
)

print(f"TFdict target genes: {len(base_GRN_TFdict)}")
print(f"RMST1 as regulator: {sum(1 for regs in base_GRN_TFdict.values() if 'Rmst' in regs)} genes")
print(f"RMST1 regulated by: {len(base_GRN_TFdict['Rmst'])} genes")
print(f"C13 as regulator: {sum(1 for regs in base_GRN_TFdict.values() if 'C130026L21Rik' in regs)} genes")
print(f"C13 regulated by {len(base_GRN_TFdict['C130026L21Rik'])} genes")

# Save
adata_celloracle.write_h5ad("../data/adata_celloracle.h5ad", compression="gzip")

Base GRN BEFORE filtering to CPM data genes: source     1095
target    21340
dtype: int64
    
Base GRN AFTER filtering to CPM data genes: source      809
target    18016
dtype: int64

Mandatory genes: 1792
  TFs: 809
  Rmst targets: 736
  C13 targets: 749
  lncRNAs: 2
Extra slots for HVGs: 308


/Users/jaime/miniforge3/envs/celloracle_env/lib/python3.10/site-packages/pandas/core/util/hashing.py:357: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding
/Users/jaime/miniforge3/envs/celloracle_env/lib/python3.10/site-packages/pandas/core/util/hashing.py:357: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding
/Users/jaime/miniforge3/envs/celloracle_env/lib/python3.10/site-packages/pandas/core/util/hashing.py:357: RuntimeWarning: invalid value encountered in cast
  vals.astype(str).astype(object), hash_key, encoding


HVG extra genes selected: 308
Final gene set size: 2100
  C130026L21Rik in final set: True
  Rmst in final set: True
adata_celloracle shape: (1604, 2100)
UMAP present: True
Leiden present: True
TFdict target genes: 1941
RMST1 as regulator: 736 genes
RMST1 regulated by: 807 genes
C13 as regulator: 749 genes
C13 regulated by 807 genes


## CellOracle object creation

Initialize CellOracle and input data

In [4]:
# CellOracle strictly expects a layer named 'raw_count' for an internal QC function,
# even when we tell it to use normalized counts. We bypass this bug by 
# copying our .X data into this specific layer name.
adata_celloracle.layers['raw_count'] = adata_celloracle.X.copy()

## Initialize CellOracle object
oracle = co.Oracle()

## Input scRNAseq data and metadata
oracle.import_anndata_as_normalized_count(
    adata=adata_celloracle,
    cluster_column_name='leiden_annotated',
    embedding_name='X_umap'
)

## Input base GRN
oracle.import_TF_data(TFdict=base_GRN_TFdict)

KNN Imputation

In [5]:
# -- PCA --
# We DO NOT run oracle.perform_PCA().
# We inject our rigorously calculated PCA (Log-norm + Z-score + Blacklist)
# This provides the true biological manifold for KNN diffusion.
oracle.pcs = adata_celloracle.obsm['X_pca'].copy()

# Important PCs (from previous analysis)
n_comps = 13 

# Determine optimal K (radius of diffusion) based on cell count
n_cell = oracle.adata.shape[0]
k = int(0.025*n_cell)
print(f"Total cells: {n_cell} | Selected neighbors (K): {k}")

# Run imputation using our custom PCA
oracle.knn_imputation(
    n_pca_dims=n_comps, 
    k=k, 
    balanced=True, 
    b_sight=k*8,
    b_maxl=k*4, 
    n_jobs=-1
)

Total cells: 1604 | Selected neighbors (K): 40


Save CellOracle objecto before GRN inference:

In [6]:
oracle.to_hdf5("../data/celloracle_data/celloracle_unfit.celloracle.oracle")